# Generated (jax-cfd) vs reference `kf_2d` — Re=1000

Both datasets are **physical units** (raw vorticity, unnormalized).

**Note:** the generated trajectories use random initial conditions and chaotic dynamics, so a given
frame index does **not** correspond to the same physical state as the reference. Compare the **flow
character** and **statistics**, not pixel-for-pixel. Select the **Python (venv-ddpm)** kernel, Run All.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "data_generation" else os.getcwd()
ref = np.load(os.path.join(ROOT, "flow-data", "kf_2d_re1000_256_40seed.npy"), mmap_mode="r")
gen = np.load(os.path.join(ROOT, "data_generation", "jaxcfd_re1000_1024to256_2seq.npy"))
print("reference kf_2d :", ref.shape)
print("generated jaxcfd:", gen.shape)

In [ ]:
# --- global statistics (physical units) ---
# Reference target = FULL kf_2d (memory-safe strided frame sample). Per-sequence std varies
# a lot in kf_2d (4.4-4.9), so comparing against only the first few seqs would be misleading.
ref_sample = np.asarray(ref[:, ::8])   # every 8th frame, all 40 seqs
def st(a):
    return a.mean(), a.std(), a.min(), a.max()
rm, rs, rmin, rmax = st(ref_sample)
gm, gs, gmin, gmax = st(gen)
print(f"{'':16s}{'mean':>10s}{'std':>10s}{'min':>10s}{'max':>10s}")
print(f"{'reference (all)':16s}{rm:10.4f}{rs:10.4f}{rmin:10.2f}{rmax:10.2f}")
print(f"{'generated':16s}{gm:10.4f}{gs:10.4f}{gmin:10.2f}{gmax:10.2f}")
print(f"\nmean difference (gen - ref) = {gm - rm:+.5f}")
print(f"std ratio (gen / ref)       = {gs / rs:.4f}   (1.0 = identical spread)")

# per-sequence std: shows the natural spread in the reference vs the generated seqs
ref_pseq = np.array([np.asarray(ref[i]).std() for i in range(ref.shape[0])])
gen_pseq = gen.std(axis=(1, 2, 3))
print(f"\nreference per-seq std: min={ref_pseq.min():.3f} max={ref_pseq.max():.3f} mean={ref_pseq.mean():.3f}")
print(f"generated per-seq std: {gen_pseq.round(3)}")

In [ ]:
# --- same-frame side-by-side (different realizations; compare character) ---
ref_seq, gen_seq = 0, 0
frames = np.linspace(0, gen.shape[1] - 1, 5, dtype=int)
vmax = float(np.percentile(np.abs(np.asarray(ref[ref_seq, frames])), 99))

fig, axes = plt.subplots(2, len(frames), figsize=(3.1 * len(frames), 6.4))
for j, fr in enumerate(frames):
    axes[0, j].imshow(ref[ref_seq, fr], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    axes[1, j].imshow(gen[gen_seq, fr], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    axes[0, j].set_title(f"frame {fr}")
    for ax in (axes[0, j], axes[1, j]):
        ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_ylabel(f"reference seq {ref_seq}", fontsize=12)
axes[1, 0].set_ylabel(f"generated seq {gen_seq}", fontsize=12)
fig.suptitle("Same frame indices — different realizations (compare character, not pixels)", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# --- statistics over time + vorticity distribution ---
r0 = np.asarray(ref[0]); g0 = gen[0]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
ax[0].plot(r0.std(axis=(-1, -2)), label="reference"); ax[0].plot(g0.std(axis=(-1, -2)), label="generated")
ax[0].set_title("per-frame std"); ax[0].set_xlabel("frame"); ax[0].grid(alpha=0.3); ax[0].legend()
ax[1].plot(r0.mean(axis=(-1, -2)), label="reference"); ax[1].plot(g0.mean(axis=(-1, -2)), label="generated")
ax[1].set_title("per-frame mean"); ax[1].set_xlabel("frame"); ax[1].grid(alpha=0.3); ax[1].legend()
bins = np.linspace(-18, 18, 120)
ax[2].hist(r0.ravel(), bins=bins, density=True, histtype="step", label="reference")
ax[2].hist(g0.ravel(), bins=bins, density=True, histtype="step", label="generated")
ax[2].set_yscale("log"); ax[2].set_title("vorticity PDF"); ax[2].set_xlabel("vorticity"); ax[2].legend()
plt.tight_layout(); plt.show()

## In-distribution test — low-res reconstruction MSE

Same sparse + NN-fill task run on the **2 simulated sequences** (input = sparse/NN-fill of the clean simulated flow, GT = clean simulated) vs the **real test set** (seqs 36–39). Same checkpoint, K=3 / S=[150,100,50], std=4.7988.

If the model reconstructs the simulated flow with MSE in the same range as the real test data, the simulated flow is **in-distribution**. (Run `python -m src.run_sim_inference` first to produce the `sim_seq*` pickles.)

In [ ]:
import pickle

RDIR = os.path.join(ROOT, "monitoring", "sequence_reconstructions")
def load_mse(name):
    r = pickle.load(open(os.path.join(RDIR, name), "rb"))
    return np.array([fr["mse"] for fr in r["frames"]])

test = {s: load_mse(f"sequence_reconstruction_seq{s}.pkl") for s in [36, 37, 38, 39]}
sim  = {s: load_mse(f"sequence_reconstruction_sim_seq{s}.pkl") for s in [0, 1]}

fig, ax = plt.subplots(figsize=(12, 5))
for s, mse in test.items():
    ax.plot(mse, color="0.65", lw=1.0, alpha=0.8, label=f"test seq{s} ({mse.mean():.3f})")
for s, mse in sim.items():
    ax.plot(mse, lw=1.8, label=f"SIM seq{s} ({mse.mean():.3f})")
ax.set_xlabel("frame index"); ax.set_ylabel("reconstruction MSE (normalized units)")
ax.set_title("Low-res reconstruction MSE — simulated vs real test set")
ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

test_all = np.concatenate(list(test.values()))
sim_all  = np.concatenate(list(sim.values()))
test_means = np.array([m.mean() for m in test.values()])
print(f"test-set  mean MSE = {test_all.mean():.4f}  (per-seq range {test_means.min():.3f}-{test_means.max():.3f})")
print(f"simulated mean MSE = {sim_all.mean():.4f}  (seqs {[round(float(m.mean()),3) for m in sim.values()]})")
verdict = "IN-distribution" if sim_all.mean() <= test_means.max() else "possibly OUT-of-distribution"
print(f"=> simulated reconstruction MSE is {verdict} relative to the real test set")

In [ ]:
# --- animated reconstruction of a simulated sequence (input | GT | reconstruction) ---
from matplotlib import animation
from IPython.display import HTML
plt.rcParams["animation.embed_limit"] = 80  # MB

SIM_SEQ = 0          # which simulated sequence to animate (0 or 1)
CH, STEP = 1, 3      # middle triplet channel; subsample frames to keep the HTML light

r = pickle.load(open(os.path.join(RDIR, f"sequence_reconstruction_sim_seq{SIM_SEQ}.pkl"), "rb"))
frames = r["frames"]
inp = np.stack([f["input"][..., CH] for f in frames])
gt  = np.stack([f["ground_truth"][..., CH] for f in frames])
rec = np.stack([f["final"][..., CH] for f in frames])
vmax = float(np.percentile(np.abs(gt), 99))

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.9))
data = [inp, gt, rec]
ims = []
for ax, title, d in zip(axes, ["input (sparse + NN-fill)", "ground truth (simulated)", "reconstruction"], data):
    im = ax.imshow(d[0], cmap="RdBu_r", vmin=-vmax, vmax=vmax, animated=True)
    ax.set_title(title); ax.axis("off"); ims.append(im)
sup = fig.suptitle("")
fig.tight_layout()

def update(f):
    for im, d in zip(ims, data):
        im.set_array(d[f])
    sup.set_text(f"Simulated seq {SIM_SEQ} (normalized) - frame {f}/{len(inp) - 1}")
    return ims

ani = animation.FuncAnimation(fig, update, frames=range(0, len(inp), STEP), interval=120, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())